# Hard Negative Mining.py

This script contains the code needed to perform Hard Negative Mining on a pre-prepared LOINC dataset. Hard Negative Mining, or HNM, is the process of searching a set of established class labels for members that are "almost, but not quite right" fits. These "hard negatives," so-labeled because they are literally difficult for a machine algorithm to classify, are used to improve LLM fine-tuning by forcing the model to learn to differentiate truly correct answers from "nearly correct" but ultimately wrong answers.

The hard negative mining algorithm is roughly as follows:

* Take as input a bunch of examples of the form `(standardized LOINC code, nonstandard input)`. The standardized code is called the _anchor_ and the nonstandard input is called the _positive_ element.
* Use an embedding function to turn all the anchors and positives into vectors.
* For each anchor, do a similarity search on the set of all positives to get back a big list of candidate options. If the positive that was originally supplied with the anchor in the dataset is one of these neighbors (it probably is), remove it from the candidates being considered (because it's obviously not a negative).
* For each of the candidates, score them on how similar they are to the anchor, then sort them in descending order, with most similar being rank 1.
* Apply hyperparameter rules (discussed in detail below) to eliminate ineligible candidates, then select the best negative from the ones remaining.

When the mining procedure completes it generates reconstructed triplets of the form `(standardized anchor string, nonstandard positive input, nonstandard negative input)` and returns the collection in a dataset for saving.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

Additionally, the `Datasets` package used to prepare and batch the training tuples for mining requires a particular optimization back-end; we need to pin specific versions here so that we can get the right compatibility.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec datasets 'transformers>=4.51.1' 'accelerate>=0.26.0'

For dependency resolution reasons, we need to install the FAISS library on a separate line to ensure CUDA GPU compatibility. Normally, we wouldn't need to list this on its own and could concatenate it with the prior `pip` statement, but `faiss-cpu` as well as all related and derived packages have had spotty development the past few years. They now have fewer pre-built wheels and do not automatically link CUDA dependencies during building. The most convenient way to overcome this is to just install this package on its own line.

In [ ]:
pip install faiss-cpu

Also, `faiss` is an ungodly finicky library that demands constant hand-holding to not crash because you sneezed in its general direction. To avoid import errors during negative mining later, we need to instantiate some manual objects here, including a pointer to an index, and then verify that a module property called `IndexFlatIP` is reachable.

If the library installed via the above `pip` command but this assertion cell doesn't work, try the following troubleshooting:

1. perform a full uninstall of any and all `faiss` libraries: `pip uninstall --yes faiss faiss-cpu faiss-gpu faiss-gpu-cu12 faiss-gpu-cu11`. This command is overkill but will forcibly remove and unlink any dependencies that may have snuck into the environment.
2. clear the pip cache: `pip cache purge`
3. reinstall _only_ the faiss cpu library: `pip install faiss-cpu`

Once this process has been executed once, the compute instance should be set going forward, but if it isn't, stop the kernel, restart it, then try this again. It's annoying, but turning the thing off then on is literally the package's recommended github distribution solution.


In [ ]:
import faiss
index = faiss.IndexFlatL2(4)
assert dir(faiss.IndexFlatIP)

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Authenticate to Key Vault
credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

# Define the workspace subscription and resources so we can instantiate a secure client
SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

# NOTE: Even though we're not directly calling any of the client functions, we do still need
# the object. Having a client instantiated acts as an authenticated connection for our compute
# instance to connect to the workspace.
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Performing negative-mining is only feasible with a compute instance attached to GPU. This cell ensures that the GPU is available for CUDA optimization. We recommend using the A100 series for the best balance of cost and performance; the NC24_ads_A100 family is ideal for these calculations.

In [ ]:
import torch
assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC storage container. Any `.txt` or `.csv` files just need to be uploaded to the desired directory within the DIBBs storage container. Unlike with more complex file types, there is no need to manually turn these files into Azure Data Assets before using them. They can simply be directly loaded from storage once they're uploaded.

In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Instantiate Negative Mining Embedder Model

Negative mining relies on using a passed-in model to create embeddings for in-batch searches.

**Important:** If you're training a Retriever (i.e. training the embedding model that will perform Approximate Nearest Neighbor search), this model should _not_ be the model that the data will eventually be used to train. _However_, if you're training a Reranker (the secondary model that sorts and re-scores the search results returned from ANN), then this model _should_ be the embedding model you will use to perform the search in production. This is so that the Reranker training will accurately reflect the properties of and clusters of vectors returned by searching.

During our extensive testing, we found that using negative mining to train the Retriever model led to reduced performance. Because the Retriever is only performing an Approximate search, it's more important for the retriever to find the right cluster center. This allows it to more accurately find dense neighborhoods of good candidate standardizations. Negative mining overly tunes the retriever to discriminating signals, which at this stage lead to quality candidates being prematurely excluded. For this reason, we do not recommend using HNM during retriever training.

Given this recommendation, the best use case for HNM is creating a strong dataset for reranker training. As a result, the best model to choose as an embedding function is an instance of the e5-large model that has been domain adapted with TSDAE _and_ task adapted with specific fine-tuning. 

The full name of our recommended version of Ellen is **intfloat_e5-large-v2_0.3_1e05_mnrl_tuned_450000_1e06_no_cn**.


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "intfloat_e5-large-v2_0.3_1e05_mnrl_tuned_450000_1e06_no_cn"
TRAINED_DIR = "fine_tuned/"

import os

# First, check if the model exists locally--if it does, nothing to do here
if os.path.exists(EMBEDDING_MODEL):
    print("Model exists locally, loading it...")
else:   
    if fs.exists("models/" + TRAINED_DIR + EMBEDDING_MODEL):
        print("Found trained model, loading from remote...")
        fs.get("models/" + TRAINED_DIR + EMBEDDING_MODEL, '.')
        print("Model loaded to local memory.")
    else:
        print("Could not find model at specified path.")
        print(
            "Check model name (esp. parameter numbers, underscores, and dashes) and fetch directory."
        )
    
print("Instantiating language model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

## Step 3: Prepare Data for Negative Mining

Using our file mount, we can read a list of positive pairs from Azure Blob Storage to use as training data. We decode the bytestrings into proper UTF-8, then read all example pairs into a list. These examples will be split out into their respective _anchor codes_, which are correct standardized LOINCs, and _positive codes_, which have nothing to do with positive or negative tests and are instead nonstandard versions of the anchor codes that the model should learn to map (hence, they belong to the "positive class").

After data is appropriately loaded and partitioned, we create a Dataset object that `sentence-transformers` can read during training batching.

In order to maximize the quality of negatives found and ensure the most comprehensive coverage of the datset, we won't use any sub-sampling here. We'll negative mine everything to find the best near-misses we can.


In [ ]:
import random
from datasets import Dataset

FINE_TUNING_DATA_FILE = "prod_emulated_reranker_pairs_no_cn.txt"

# If this is set to true, then the first column of the positive pairs data set
# is assumed to be the actual numeric LOINC code, meaning the column indices to
# extract will all get bumped by one
FIRST_COLUMN_IS_CODE = False
ANCHOR_COL = 0
POSITIVE_COL = 1
if FIRST_COLUMN_IS_CODE:
    ANCHOR_COL += 1
    POSITIVE_COL += 1

examples = []

print("Loading fine-tuning positive pairs...")
with fs.open(FINE_TUNING_DATA_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            examples.append(line_str.strip())

# Sub-divide into anchor codes and nonstandard input codes
anchor_codes = []
positive_codes = []
for ex in examples:
    pair = ex.split("|")
    anchor_codes.append(pair[ANCHOR_COL].strip())
    positive_codes.append(pair[POSITIVE_COL].strip())

# Prepare the dataset object for batching
dataset = Dataset.from_dict({
    "anchor": anchor_codes,
    "positive": positive_codes
})

## Step 4: Perform HNM

This cell executes the negative mining step proper. `sentence-transformers` provides solid documentation about the function in the research paper that developed it, accessible from the [function docs](https://sbert.net/docs/package_reference/util.html#sentence_transformers.util.mine_hard_negatives). There are a number of important parameters that influence the function of HNM:

* `range_min`: The smallest rank within the list of negatives that is allowed to be considered as a possible candidate. When the model considers a particular anchor, it vector searches all the positive elements in the Dataset _except_ the positive element already associated with the anchor. It scores each positive element, then sorts them in a ranked list. A lower rank in this list means that the possible negative is closer to the anchor (e.g. rank 1 means the candidate is the most similar string to the anchor in the entire dataset except for the positive element already in the tuple). This parameter is a cutoff that forces the model to only look at rank `X` or higher. This is useful for weeding out possible negatives that are either too similar to the anchor (and thus inseparable by a machine classifier), or are more similar to the anchor than the existing positive element.
<br></br>
* `range_max`: The largest rank in the list of sorted negatives that is allowed to be considered a candidate. A larger value of this parameter means the negative mining algorithm will look at more possible candidates before deciding the best negative. This might lead to better quality negatives, especially those that respect the other hyperparameters, but it also slows down processing time, so balance here is key. If this is set to `None`, the mining algorithm will automatically estimate a value for it before performing searches (typically 100 or less).
<br></br>
* `relative_margin`: The maximum similarity **distance** between the anchor and a candidate negative expressed as a proportion of the similarity score between the anchor and the positive element. The negative mining algorithm will only consider candidate negatives whose similarity to the anchor is less than `1 - X` times the similarity of the positive element to the anchor. For example, say the margin is 0.1. If a particular (anchor, positive) pair has a similarity of 0.9, then the negative mining algorithm will look for negative candidates whose similarity to the anchor is at most (1.0 - 0.1) * 0.9 = 0.9 * 0.9 = 0.81. A higher value for this parameter allows the candidate negatives to get "closer" to both the anchor and to the positive element, which creates negatives that are harder to differentiate. This can improve fine-tuning, but only up to a point: if negatives are too similar, the model becomes unable to tell them apart from positives. This parameter, along with `range_max` above, is the chief determinant of whether negatives will be found for a given anchor. If it is set too low, or if `range_max` is too low, then it is likely that some anchors will not produce negatives. When the algorithm finishes and reports on the percentage of negatives that were "skipped," consider adjusting these values if this percentage is too high.
<br></br>
* `negatives_per_anchor`: The number of new triplets to generate for each (anchor, positive) pair in the original dataset. Each generated triplet will have a unique negative found during mining. For Text to Code purposes, we want this value to be 1. This ensures that we don't over-represent the LOINC codes for which negative mining is easy, while also making sure our training set is still comprehensive and robust.
<br></br>
* `negative_embedding_batch_size`: The batch size of the embedding portion of the negative miner. Before doing any negative searches, all of the anchor codes must be vectorized for similarity comparisons. This parameter simply governs how many anchors are batched at once. It has no impact on negative quality or selection. Values as low as 32 and as high as 1024 can work fine, so long as the compute instance has the memory to support it.

When the procedure is finished, a chart will be displayed indicating the number of negatives found, as well as any that had to be skipped for margin reasons. The distribution of negative similarity scores will also be shown so that the values can be adjusted for subsequent runs.

**Note:** This process takes a _long_ time. Average runs of negative mining with a full GPU suite can take 30-60 minutes to conclude. Make sure you don't disconnect from the notebook or enter sleep mode during that time!


In [ ]:
from sentence_transformers.util import mine_hard_negatives

RANGE_MIN = 0
RANGE_MAX = 50
RELATIVE_MARGIN = 0.025
NEGATIVES_PER_ANCHOR = 5
NEGATIVE_EMBEDDING_BATCH_SIZE = 1024

OUTPUT_FORMAT = "labeled-list"

DATASET_SAVE_NAME = "hnm_prod_emulated_reranker_pairs_no_cn_1_5_margin_025"

hard_mined_dataset = mine_hard_negatives(
    dataset=dataset,
    model=embedding_model,
    range_min=RANGE_MIN,
    range_max=RANGE_MAX,
    relative_margin=RELATIVE_MARGIN,
    num_negatives=NEGATIVES_PER_ANCHOR,
    sampling_strategy="top",
    batch_size=NEGATIVE_EMBEDDING_BATCH_SIZE,
    use_faiss=True,
    output_format=OUTPUT_FORMAT,
    output_scores=False
)

hard_mined_dataset.save_to_disk(DATASET_SAVE_NAME)

Once the mining procedure completes, the data will be saved in a local dataset for working with in other notebooks. To save a copy to Azure Remote Storage, run the cell below, specifying directories appropriately.

In [ ]:
REMOTE_DATA_DIRECTORY = "hnm_data/"

# NOTE: We need the extra '/' here because the HNM dataset is a directory
fs.put(DATASET_SAVE_NAME + "/", REMOTE_DATA_DIRECTORY)

Finally, knowing how many rows exist in the dataset is useful for future tasks like fine-tuning, where we need to manually partition the data into training and validation sets. A simple print statement here allows us to get a quick glance at how many rows of data there are.

In [ ]:
print(hard_mined_dataset)